#imports

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable
from pyspark.sql.utils import AnalysisException

# Configuration : Utilities

In [0]:
# Create notebook widget for Catalog
dbutils.widgets.text("catalog", "lakehouse_stocks", "Catalog")
catalog = dbutils.widgets.get("catalog")

In [0]:
# Define Bronze source tables (Inputs)
bronze_twelvedata_table = f"{catalog}.bronze.twelvedata_raw"
bronze_fmp_table = f"{catalog}.bronze.fmp_raw"


In [0]:
# Define Silver target tables (Outputs)
silver_prices_table = f"{catalog}.silver.stock_prices"
silver_company_table = f"{catalog}.silver.company_profiles"

In [0]:
print(f"Reading Bronze: {bronze_twelvedata_table}, {bronze_fmp_table}")
print(f"Writing Silver: {silver_prices_table}, {silver_company_table}")

# Cleaning and Deduplicating Twelve Data

# also implement High-Water Mark + UPSERT

In [0]:
## High-Water Mark: It dynamically queries the Silver table to find the exact timestamp of the last processed record, uses it to filter the ## ## Bronze data before applying transformations, and surgically merges only the new rows.

try:
    watermarks_df = spark.sql(
        f"select MAX(ingested_at) as max_val from {silver_prices_table}")

    high_water_mark = watermarks_df.collect()[0]["max_val"]
except AnalysisException:
    # Table not exists yet.
    high_water_mark = None

#  Read from Bronze, applying the filter EARLY to prevent a full table scan
df_bronze_prices = spark.table(bronze_twelvedata_table)

if high_water_mark:
    df_bronze_prices = df_bronze_prices.filter(
        F.col("ingested_at") > high_water_mark)



In [0]:
# define_Window
price_window = Window.partitionBy("symbol","datetime")\
    .orderBy(F.col("ingested_at").desc())

In [0]:
# Apply transformation, filtering, and casting
df_silver_prices = (
    df_bronze_prices
    .withColumn("rn", F.row_number().over(price_window))
    .filter(F.col("rn") == 1)
    .select(
        F.col("symbol").cast("string"),
        F.col("datetime").cast("date").alias("price_date"),
        F.col("open").cast("double"),
        F.col("high").cast("double"),
        F.col("low").cast("double"),
        F.col("close").cast("double"),
        F.col("volume").cast("long"),
        F.col("ingested_at")
    )
)

print(f"Cleaned Silver Prices Count: {df_silver_prices.count()}")
display(df_silver_prices.limit(5))

In [0]:
# Clean & Deduplicate FMP (Company Profiles)
company_window = Window.partitionBy("symbol")\
    .orderBy(F.col("ingested_at").desc())

In [0]:
## High-Water Mark: It dynamically queries the Silver table to find the exact timestamp of the last processed record, uses it to filter the ## ## Bronze data before applying transformations, and surgically merges only the new rows.

try:
    watermarks_df = spark.sql(
        f"select MAX(ingested_at) as max_val from {silver_company_table}")

    high_water_mark = watermarks_df.collect()[0]["max_val"]
except AnalysisException:
    # Table not exists yet.
    high_water_mark = None

#  Read from Bronze, applying the filter EARLY to prevent a full table scan
df_bronze_company = spark.table(bronze_fmp_table)

if high_water_mark:
    df_bronze_company = df_bronze_company.filter(
        F.col("ingested_at") > high_water_mark)



In [0]:
# Apply transformation, filtering, and snake_case standardization
df_silver_company = (
    df_bronze_company
    .withColumn("rn", F.row_number().over(company_window))
    .filter(F.col("rn") == 1)
    .select(
        F.col("symbol").cast("string"),
        F.col("companyName").cast("string").alias("company_name"),
        F.col("sector").cast("string"),
        F.col("industry").cast("string"),
        F.col("exchange").cast("string"),
        F.col("exchangeFullName").cast("string").alias("exchange_full_name"),
        F.col("country").cast("string"),
        F.col("currency").cast("string"),
        F.col("ipoDate").cast("date").alias("ipo_date"),
        F.col("cik").cast("string"),
        F.col("isin").cast("string"),
        F.col("cusip").cast("string"),
        F.col("ingested_at")
    )
)

print(f"Cleaned Silver Company Profiles Count: {df_silver_company.count()}")
display(df_silver_company.limit(5))

In [0]:
df_silver_company.printSchema()

In [0]:

# Define exact S3 paths inside your processed bucket
s3_silver_prices_path = "s3://modern-lakehouse-processed/silver/stock_prices/"
s3_silver_company_path = "s3://modern-lakehouse-processed/silver/company_profiles/"

if not spark.catalog.tableExists(silver_prices_table):
    #  Create the table for the first time
    (
        df_silver_prices.write
        .format("delta")
        .option("path", s3_silver_prices_path)
        .saveAsTable(silver_prices_table)
    )
    print(f"Created initial Silver table: {silver_prices_table}")
else:
    # If akready existed Perform the UPSERT
    delta_prices = DeltaTable.forName(spark, silver_prices_table)
    (
        delta_prices.alias("target")
        .merge(
            df_silver_prices.alias("source"),
            "target.symbol = source.symbol AND target.price_date = source.price_date"
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )
    print(f"Successfully merged new data into: {silver_prices_table}")

# for Company Profiles
if not spark.catalog.tableExists(silver_company_table):
    # Create the table for the first time
    (
        df_silver_company.write
        .format("delta")
        .option("path", s3_silver_company_path)
        .saveAsTable(silver_company_table)
    )
    print(f"Created initial Silver table: {silver_company_table}")
else:
    # If already existed Perform  UPSERT
    delta_company = DeltaTable.forName(spark, silver_company_table)
    (
        delta_company.alias("target")
        .merge(
            df_silver_company.alias("source"),
            "target.symbol = source.symbol"
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )
    print(f"Successfully merged new data into: {silver_company_table}")

In [0]:
#  Instantiate the DeltaTable API for the Silver tables
delta_prices = DeltaTable.forName(spark, silver_prices_table)
delta_company = DeltaTable.forName(spark, silver_company_table)

#  Display the transaction log history for the prices table
print(f"Transaction History for {silver_prices_table}:")
display(delta_prices.history().select("version", "timestamp", "operation", "operationParameters").limit(5))

In [0]:
display(spark.sql(f"DESCRIBE DETAIL {silver_prices_table}").select("location"))


In [0]:
display(spark.sql(f"DESCRIBE DETAIL {silver_company_table}").select("location"))